In [1]:
from irrigator.forecasts.arome_processing import (
    sync_static,
    sync_forecast,
    sync_dynamic,
    get_available_coverages,
    parse_coverage_id,
    find_available_run_dates,
    load_arome_daily_cache,
    ensure_dirs,
    DAILY_DIR,
    RAW_DIR,
)
from collections import defaultdict
import os

REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)


results = sync_static(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

results_dyn = sync_dynamic(overwrite=False)


results = sync_forecast(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

    

2026-08-16 18:51:11,807 Syncing static archive...
2026-08-16 18:53:39,516 [static] 2026-08-15: 98 calls
2026-08-16 18:53:44,259 [static] arome_daily_2026-08-15.nc → 24.5 MB
2026-08-16 18:56:04,563 [static] 2026-08-16: 98 calls
2026-08-16 18:56:07,625 [static] arome_daily_2026-08-16.nc → 24.3 MB
2026-08-16 18:56:07,626 Sync complete: 5 available, 3 cached, 2 fetched, 0 errors
2026-08-16 18:56:07,627 Syncing dynamic archive...


  ✓ 2026-08-12: cached
  ✓ 2026-08-13: cached
  ✓ 2026-08-14: cached
  ↓ 2026-08-15: fetched
  ↓ 2026-08-16: fetched


2026-08-16 18:56:28,461 [dynamic] 2026-08-14T18: done
2026-08-16 18:56:46,767 [dynamic] 2026-08-14T21: done
2026-08-16 18:57:09,629 [dynamic] 2026-08-15T00: done
2026-08-16 18:57:30,771 [dynamic] 2026-08-15T03: done
2026-08-16 18:57:49,421 [dynamic] 2026-08-15T06: done
2026-08-16 18:58:10,360 [dynamic] 2026-08-15T09: done
2026-08-16 18:58:29,166 [dynamic] 2026-08-15T12: done
2026-08-16 18:58:48,509 [dynamic] 2026-08-15T15: done
2026-08-16 18:59:10,247 [dynamic] 2026-08-15T18: done
2026-08-16 18:59:29,588 [dynamic] 2026-08-15T21: done
2026-08-16 18:59:48,712 [dynamic] 2026-08-16T00: done
2026-08-16 19:00:10,507 [dynamic] 2026-08-16T03: done
2026-08-16 19:00:35,694 [dynamic] 2026-08-16T06: done
2026-08-16 19:00:55,087 [dynamic] 2026-08-16T09: done
2026-08-16 19:01:13,559 [dynamic] 2026-08-16T12: done
2026-08-16 19:01:17,165 [dynamic] arome_daily_2026-08-12.nc → 8 windows, 23.3 MB
2026-08-16 19:01:22,319 [dynamic] arome_daily_2026-08-13.nc → 8 windows, 23.6 MB
2026-08-16 19:01:25,929 [dyn

  ✓ 2026-08-12: cached
  ✓ 2026-08-13: cached
  ✓ 2026-08-14: cached
  ↓ 2026-08-15: fetched
  ↓ 2026-08-16: fetched


# IFS

In [2]:
import os
import logging
from datetime import date, timedelta
from pathlib import Path
from irrigator.ingestion.ifs_ens_client import run_ifs_pipeline, load_ifs_daily

# Today only (default)
# daily_paths = run_ifs_pipeline()

# Or a date range:
daily_paths = run_ifs_pipeline(
    start=date(2026, 8, 5),
    end=date.today(),
    keep_raw=False,  # delete GRIBs after processing (default)
    #split_params=False
)

print(f"\nProcessed {len(daily_paths)} runs:")
for p in daily_paths:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.0f} MB)")


2026-08-16 19:11:03,606 IFS ENS 2026-08-05 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-05_00z.nc


2026-08-16 19:11:35,300 IFS ENS 2026-08-06 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-06_00z.nc
2026-08-16 19:12:06,969 IFS ENS 2026-08-07 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-07_00z.nc
2026-08-16 19:12:38,683 IFS ENS 2026-08-08 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-08_00z.nc
2026-08-16 19:13:08,684 IFS ENS 2026-08-09 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-09_00z.nc
2026-08-16 19:13:40,494 IFS ENS 2026-08-10 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-10_00z.nc
2026-08-16 19:14:12,277 IFS ENS 2026-08-11 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-11_00z.nc
2026-08-16 19:14:44,050 IFS ENS 2026-08-12 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-12_00z.nc
2026-08-16 19:15:15,809 IFS ENS 2026-08-13 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-13_00z.nc
2026-08-16 19:15:47,563 IFS ENS 2026-08-14 00Z already processed

<multiple>:   0%|          | 0.00/16.2G [00:00<?, ?B/s]

2026-08-16 19:38:02,306 Downloaded IFS ENS: data/raw/ifs_ens/ifs_ens_2026-08-15_00z.grib2 (17375.6 MB)
2026-08-16 19:38:02,307 [2026-08-15 00Z] Opening and slicing to bbox...


By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.


2026-08-16 19:39:09,441 [2026-08-15 00Z] Processing to daily...
2026-08-16 19:46:04,642 IFS ENS daily: 16 days, 50 members, 8 variables
2026-08-16 19:46:06,471 Saved IFS ENS daily: data/processed/ifs_ens/ifs_daily_2026-08-15_00z.nc (47.7 MB, 50 members, 16 days)
2026-08-16 19:46:07,682 [2026-08-15 00Z] Deleted raw GRIB (17376 MB freed)
2026-08-16 19:46:39,538 [2026-08-16 00Z] Downloading...
2026-08-16 19:46:39,539 Downloading IFS ENS (bulk, source=ecmwf): 2026-08-16 00Z, 61 steps, 7 params
2026-08-16 19:47:03,136 Downloading <multiple>


<multiple>:   0%|          | 0.00/16.1G [00:00<?, ?B/s]

2026-08-16 20:08:05,449 Downloaded IFS ENS: data/raw/ifs_ens/ifs_ens_2026-08-16_00z.grib2 (17300.9 MB)
2026-08-16 20:08:05,450 [2026-08-16 00Z] Opening and slicing to bbox...
2026-08-16 20:09:17,294 [2026-08-16 00Z] Processing to daily...
2026-08-16 20:16:09,702 IFS ENS daily: 16 days, 50 members, 8 variables
2026-08-16 20:16:11,441 Saved IFS ENS daily: data/processed/ifs_ens/ifs_daily_2026-08-16_00z.nc (48.1 MB, 50 members, 16 days)
2026-08-16 20:16:12,368 [2026-08-16 00Z] Deleted raw GRIB (17301 MB freed)
2026-08-16 20:16:12,377 IFS pipeline complete: 12/12 runs processed



Processed 12 runs:
  ifs_daily_2026-08-05_00z.nc  (46 MB)
  ifs_daily_2026-08-06_00z.nc  (46 MB)
  ifs_daily_2026-08-07_00z.nc  (46 MB)
  ifs_daily_2026-08-08_00z.nc  (46 MB)
  ifs_daily_2026-08-09_00z.nc  (46 MB)
  ifs_daily_2026-08-10_00z.nc  (46 MB)
  ifs_daily_2026-08-11_00z.nc  (47 MB)
  ifs_daily_2026-08-12_00z.nc  (47 MB)
  ifs_daily_2026-08-13_00z.nc  (47 MB)
  ifs_daily_2026-08-14_00z.nc  (47 MB)
  ifs_daily_2026-08-15_00z.nc  (48 MB)
  ifs_daily_2026-08-16_00z.nc  (48 MB)


# SEAS5 hindcast archive (one-time per initialization month)

The seasonal bias correction needs the ECMWF SEAS5 1993–2016 retrospective
forecasts for the same initialization month as the operational forecast. The
download is intentionally opt-in because it is a sizeable one-time archive.

You also need processed France-wide ERA5-Land daily files covering the historical
period used for analog matching; the new PCA code reads those annual files lazily.

In [ ]:
from datetime import date
from irrigator.ingestion.cds_client import fetch_seas5_hindcasts

RUN_SEAS5_HINDCAST_ARCHIVE = True
SEAS5_INIT_MONTH = date.today().month

if RUN_SEAS5_HINDCAST_ARCHIVE:
    paths = fetch_seas5_hindcasts(
        init_month=SEAS5_INIT_MONTH,
        start_year=1993,
        end_year=2016,
        raw_dir=Path("data/raw"),
        overwrite=False,
    )
    print(f"SEAS5 hindcasts ready: {len(paths)} initialization files")
else:
    print("SEAS5 hindcast archive skipped. Set RUN_SEAS5_HINDCAST_ARCHIVE=True once when needed.")


In [1]:
from irrigator.ingestion.cds_client import (
    _seas5_output_path,
    FRANCE_BBOX,
    DEFAULT_RAW_DIR,
    BBoxWGS84,
)

from pathlib import Path
from datetime import date

import numpy as np
import xarray as xr
from irrigator.ingestion.cds_client import (
    _seas5_output_path,
    FRANCE_BBOX,
    DEFAULT_RAW_DIR,
    BBoxWGS84,
)
from pathlib import Path
import xarray as xr

from datetime import date
import numpy as np


def date_to_datetime64(d: date) -> np.datetime64:
    return np.datetime64(d, "ns")


def convert_seas5_monthly_accumulations(ds: xr.Dataset) -> xr.Dataset:
    """Convert SEAS5 monthly mean rates/fluxes to monthly accumulations.

    Converts
    --------
    tprate : m s-1
        -> tp : m accumulated over the forecast month

    msdsrf : W m-2
        -> ssrd : J m-2 accumulated over the forecast month

    The actual forecast month is determined from:
        forecast_reference_time + (forecastMonth - 1) months

    Thus the conversion automatically accounts for:
    - 28, 29, 30 and 31-day months
    - leap years
    - year transitions
    """

    

    seconds_in_day =  24 * 60 * 60

    # DataArray so xarray automatically broadcasts over
    # number / latitude / longitude



    # m s-1 -> m month-1
    ds["tp"] = ds["tprate"] * seconds_in_day
    ds["tp"].attrs = ds["tprate"].attrs.copy()
    ds["tp"].attrs.update(
        {
            "long_name": "Average daily Total precipitation accumulated over forecast month",
            "units": "m",
        }
    )

    # W m-2 = J s-1 m-2 -> J m-2 month-1
    ds["ssrd"] = ds["msdsrf"] * seconds_in_day
    ds["ssrd"].attrs = ds["msdsrf"].attrs.copy()
    ds["ssrd"].attrs.update(
        {
            "long_name": ("Average daily Surface solar radiation downwards accumulated over forecast month"),
            "units": "J m-2",
        }
    )

    # Original rates no longer needed
    ds = ds.drop_vars(["tprate", "msdsrf"])

    return ds

def fetch_seas5(
    year: int,
    month: int,
    ref_file: str,
    *,
    bounding_box: BBoxWGS84 = FRANCE_BBOX,
    raw_dir: str | Path = DEFAULT_RAW_DIR,
) -> Path:

    raw_dir = Path(raw_dir)
    out_path = _seas5_output_path(raw_dir, year, month)

    ds_ref = xr.open_dataset(ref_file)

    ds_ref = ds_ref.sel(
        forecast_reference_time=date_to_datetime64(date(year, month, 1)),
        longitude=slice(
            bounding_box.west,
            bounding_box.east,
        ),
        latitude=slice(
            bounding_box.north,
            bounding_box.south,
        ),
    )

    # Correct monthly accumulation conversion
    ds_ref = convert_seas5_monthly_accumulations(ds_ref)

    ds_ref.to_netcdf(out_path)

    return out_path


In [2]:
for archive in [1,2,3,4]:
    ref_file = f"/home/mbaldacchino/data/seas5_archive_{str(archive)}.nc"
    for year in range(1993, 2027):
        for month in range(1, 13):
            try:
                out_path = fetch_seas5(year, month, ref_file=ref_file)
                print(f"Downloaded SEAS5 {year}-{month:02d} to {out_path}")
            except: 
                continue

Downloaded SEAS5 1993-01 to data/raw/seas5/seas5_1993_01.nc
Downloaded SEAS5 1993-02 to data/raw/seas5/seas5_1993_02.nc
Downloaded SEAS5 1993-03 to data/raw/seas5/seas5_1993_03.nc
Downloaded SEAS5 1993-04 to data/raw/seas5/seas5_1993_04.nc
Downloaded SEAS5 1993-05 to data/raw/seas5/seas5_1993_05.nc
Downloaded SEAS5 1993-06 to data/raw/seas5/seas5_1993_06.nc
Downloaded SEAS5 1993-07 to data/raw/seas5/seas5_1993_07.nc
Downloaded SEAS5 1993-08 to data/raw/seas5/seas5_1993_08.nc
Downloaded SEAS5 1993-09 to data/raw/seas5/seas5_1993_09.nc
Downloaded SEAS5 1993-10 to data/raw/seas5/seas5_1993_10.nc
Downloaded SEAS5 1993-11 to data/raw/seas5/seas5_1993_11.nc
Downloaded SEAS5 1993-12 to data/raw/seas5/seas5_1993_12.nc
Downloaded SEAS5 1994-01 to data/raw/seas5/seas5_1994_01.nc
Downloaded SEAS5 1994-02 to data/raw/seas5/seas5_1994_02.nc
Downloaded SEAS5 1994-03 to data/raw/seas5/seas5_1994_03.nc
Downloaded SEAS5 1994-04 to data/raw/seas5/seas5_1994_04.nc
Downloaded SEAS5 1994-05 to data/raw/sea